In [10]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [11]:
df = pd.read_csv("../data/adult.csv", index_col=0)
df_clean = df.loc[~df.eq("?").any(axis=1)].copy()
salary_correct = (
    (df_clean["salary"].eq("<=50K") & df_clean["salary K$"].le(50)) |
    (df_clean["salary"].eq(">50K") & df_clean["salary K$"].gt(50))
)
incorrect_salary_rows = df_clean.loc[~salary_correct, ["salary", "salary K$"]]

# Task 1
Print the count of men and women in the dataset.

In [12]:
gender_count = df_clean["sex"].value_counts()
gender_count

sex
Male      20380
Female     9782
Name: count, dtype: int64

# Task 2
Find the average age of men in dataset

In [13]:
avg_male_age = (
    df_clean.loc[df_clean["sex"].eq("Male"), "age"].mean()
)
avg_male_age

np.float64(39.18400392541707)

# Task 3
Get the percentage of people from Poland (native-country)

In [14]:
poland_percentage = (
    df_clean["native-country"].eq("Poland").mean() * 100
)
round(poland_percentage, 2)

np.float64(0.19)

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [15]:
salary_age_stats = (df_clean.groupby("salary")["age"].agg(["mean", "std"]))
salary_age_stats

,mean,std
salary,,
<=50K,36.60806,13.464631
>50K,43.95911,10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [18]:
df = pd.read_csv(
    "../data/adult.csv",
    usecols=np.array(["education", "salary"], dtype="object")
)

higher_education = np.array(
    "Bachelors Prof-school Assoc-acdm Assoc-voc Masters Doctorate".split(),
    dtype="object"
)

mask = np.logical_and(
    np.logical_not(df["education"].isin(higher_education)),
    df["salary"].eq(">50K")
)

result = df.loc[mask]

result.shape[0]
result["education"].value_counts()

education
HS-grad         1675
Some-college    1387
10th              62
11th              60
7th-8th           40
12th              33
9th               27
5th-6th           16
1st-4th            6
Name: count, dtype: int64

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [24]:
age_statistics = (
    df.groupby("education")["age"]
      .describe()
)
age_statistics

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,933.0,37.429796,16.720713,17.0,22.00,34.0,52.0,90.0
11th,1175.0,32.355745,15.545485,17.0,18.00,28.0,43.0,90.0
12th,433.0,32.000000,14.334625,17.0,19.00,28.0,41.0,79.0
1st-4th,168.0,46.142857,15.615625,19.0,33.00,46.0,57.0,90.0
5th-6th,333.0,42.885886,15.557285,17.0,29.00,42.0,54.0,84.0
7th-8th,646.0,48.445820,16.092350,17.0,34.25,50.0,61.0,90.0
9th,514.0,41.060311,15.946862,17.0,28.00,39.0,54.0,90.0
Assoc-acdm,1067.0,37.381443,11.095177,19.0,29.00,36.0,44.0,90.0
Assoc-voc,1382.0,38.553546,11.631300,19.0,30.00,37.0,46.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [25]:
married_mask = df["marital-status"].str.startswith("Married")
salary_comparison = (
    df.assign(
        marital_group=np.where(
            married_mask,
            "Married",
            "Not married"
        )
    )
    .groupby("marital_group")["salary"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
)
salary_comparison

marital_group  salary
Married        <=50K     56.307972
               >50K      43.692028
Not married    <=50K     93.554596
               >50K       6.445404
Name: percentage, dtype: float64

# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [26]:
max_hours = df["hours-per-week"].max()
people_count = (
    df["hours-per-week"]
    .eq(max_hours)
    .sum()
)
print("Max hours per week:", max_hours)
print("People working these hours:", people_count)

Max hours per week: 99
People working these hours: 85


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [27]:
df["salary_num"] = df["salary"].eq(">50K").astype("int8")
numeric_df = df.select_dtypes(include=np.number)
correlation_matrix = numeric_df.corr(numeric_only=True)
salary_correlation = (
    correlation_matrix["salary_num"]
    .sort_values(ascending=False)
)

salary_correlation

salary_num        1.000000
salary K$         0.855788
age               0.234037
hours-per-week    0.229689
Name: salary_num, dtype: float64